In [3]:

from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv


In [ ]:

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    # api key
)

In [6]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [7]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [8]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [9]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [10]:
intial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': [{'type': 'text', 'text': 'Here is a detailed, SEO-friendly outline for a comprehensive blog post on the **"Rise of AI in India."** \n\n---\n\n# **Blog Title Options:**\n*   *Namaste AI: How India is Becoming a Global Artificial Intelligence Powerhouse*\n*   *The AI Revolution in India: Opportunities, Challenges, and the Road Ahead*\n*   *From IT Outsourcing to AI Innovation: The Rise of Artificial Intelligence in India*\n\n---\n\n## **I. Introduction**\n*   **Hook:** Mention India’s journey from being the back-office of the world (IT services) to becoming a front-runner in deep tech and Artificial Intelligence.\n*   **The Shift:** Briefly highlight how AI is no longer a sci-fi concept in India, but a reality integrated into daily life (UPI payments, WhatsApp chatbots, agriculture apps).\n*   **Thesis Statement:** Driven by a massive talent pool, government initiatives, and a booming startup ecosystem, India is uniquely positioned to shape th